# 나만의 물체를 찾는 드론 만들기
### YOLO11n 파인튜닝 → Raspberry Pi AI Camera(IMX500) 탑재

기본 모델은 COCO 80종(사람, 자동차, 병 등)의 사물만 감지할 수 있습니다.
본 실습에서는 **직접 수집한 사진**으로 모델을 학습시켜 원하는 물체를 찾도록 만듭니다.

**전체 흐름**

```
[Colab]  사진 수집 → 라벨링 → 학습 → 양자화 → packerOut.zip
                                                    ↓ 다운로드
[Pi]     imx500-package → network.rpk → AI 카메라에 업로드 → 추적 비행
```

> ⚠️ 마지막 `.rpk` 포장 단계는 Colab 환경에서 진행할 수 없습니다.
> 포장 도구(`imx500-tools`)가 ARM 전용이므로 **라즈베리파이에서** 실행해야 합니다.
> 이 노트북에서는 그 직전 단계인 `packerOut.zip` 생성까지 진행합니다.

**시작 전에:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 0. GPU 확인

In [ ]:
!nvidia-smi -L
print('위에 Tesla T4 같은 게 안 보이면 런타임 유형을 GPU로 바꾸세요!')

## 1. 설치

학습용(`ultralytics`)과 IMX500 변환용 패키지를 **여기서 한 번에** 설치합니다. (5분 정도)

버전은 ultralytics 가 변환 시점에 요구하는 것과 동일하게 맞춰 두었습니다.
이렇게 미리 깔아 두면 6번 변환 단계에서 패키지가 갈리며 나는 `ImportError` 를 피할 수 있습니다.

> ⚠️ 설치 중 `protobuf` 버전이 내려가면서 Colab 기본 패키지와의 충돌 경고가 여러 줄 뜹니다.
> 이 노트북에서 쓰지 않는 패키지들이므로 **무시해도 됩니다.**
> 셀 마지막에서 런타임이 **자동으로 재시작**됩니다. "세션이 다운되었습니다" 메시지도 정상입니다.
> 재시작된 뒤 그 다음 셀부터 이어서 실행하세요.

In [ ]:
# IMX500 변환기(imx500-converter)는 Java 17 이상을 필요로 합니다.
!apt-get -qq install -y openjdk-21-jre > /dev/null

# 아래 버전 조건은 ultralytics 8.4.x 의 IMX 변환이 요구하는 것과 같습니다.
!pip install -q ultralytics \
    "model-compression-toolkit>=2.4.1" "edge-mdt-cl<1.1.0" "edge-mdt-tpc>=1.2.0" \
    "pydantic<2.12" "imx500-converter[pt]>=3.17.3"

# 교체된 패키지를 반영하려면 런타임을 한 번 재시작해야 합니다.
import os
os.kill(os.getpid(), 9)

재시작이 끝나면 아래 셀로 설치를 확인하고 계속 진행하세요.

In [ ]:
import ultralytics
ultralytics.checks()

## 2. 데이터셋 준비

Roboflow에서 라벨링한 데이터셋을 내려받아 사용합니다.

1. https://roboflow.com 가입 → New Project 생성 → 프로젝트 유형으로 Object Detection 선택
2. 사진 업로드 → 바운딩 박스 라벨링 → Generate 진행
3. 좌측 Dataset 탭 → 이미지 전체 선택 → Export Data → Download Dataset → 포맷은 **YOLOv11** → Export As → **Code Snippet**
4. 나오는 코드에서 `https://app.roboflow.com/ds/...` 로 시작하는 **URL만** 복사해 아래 셀에 붙여넣으세요.

### YOLO 데이터셋 형식
```
my_dataset/
  data.yaml          <- 클래스 이름 목록
  train/images/*.jpg
  train/labels/*.txt <- 한 줄에 "클래스번호 x중심 y중심 너비 높이" (0~1 비율)
  valid/images/*.jpg
  valid/labels/*.txt
```

**필요한 이미지 수:** 클래스당 최소 100장, 권장 200~300장입니다.
드론 추적용으로 사용할 모델이라면 **위에서 내려다본 각도(탑뷰)** 및 **원거리 촬영 사진**을 반드시 포함해야 합니다.

In [ ]:
# Roboflow 다운로드 코드에서 복사한 URL 을 붙여넣으세요. (key= 뒷부분까지 전부)
RF_URL = 'https://app.roboflow.com/ds/XXXXXXXX?key=YYYYYYYY'

DATASET_DIR = '/content/dataset'

!rm -rf {DATASET_DIR} && mkdir -p {DATASET_DIR}
!curl -sL "{RF_URL}" -o /content/roboflow.zip
!unzip -q /content/roboflow.zip -d {DATASET_DIR}
!ls {DATASET_DIR}

In [ ]:
import os, glob, yaml

# 1) 압축 해제된 실제 위치 찾기 (하위 폴더로 한 겹 더 들어가는 경우가 있습니다)
DATA_YAML = sorted(glob.glob(DATASET_DIR + '/**/data.yaml', recursive=True))[0]
ROOT = os.path.dirname(DATA_YAML)
print('데이터셋 루트:', ROOT)
print('폴더 구성 :', sorted(os.listdir(ROOT)))

# 2) split 별 이미지 폴더를 실제로 찾아서 경로를 다시 씁니다.
#    (Roboflow 의 data.yaml 은 '../train/images' 라 ultralytics 가 못 찾습니다)
def find_images(*names):
    for n in names:
        for cand in (f'{n}/images', n):
            if glob.glob(os.path.join(ROOT, cand, '*.[jpJP]*')):
                return cand
    return None

train, val, test = find_images('train'), find_images('valid', 'val'), find_images('test')
assert train, f'{ROOT} 안에서 학습 이미지를 찾지 못했습니다. 폴더 구성을 확인하세요.'
if not val:
    print('⚠️ 검증(valid) 셋이 없어 train 을 검증용으로 함께 씁니다. Roboflow 에서 Train/Valid 분할을 확인하세요.')
    val = train

d = yaml.safe_load(open(DATA_YAML))
d['path'], d['train'], d['val'] = ROOT, train, val
d['test'] = test if test else None
if d['test'] is None:
    d.pop('test')
yaml.safe_dump(d, open(DATA_YAML, 'w'), allow_unicode=True, sort_keys=False)

print()
print(open(DATA_YAML).read())
for k in ('train', 'val'):
    print(k, len(glob.glob(os.path.join(ROOT, d[k], '*'))), '장')

## 3. 학습 설정

**아래 파라미터 수치들을 자유롭게 변경하며 실험해 보세요.**

In [ ]:
# 학습을 몇 바퀴 돌릴지. 적으면 덜 배우고, 너무 많으면 외워버립니다(과적합).
EPOCHS = 60

# 입력 이미지 크기. 작을수록 빠르고, 클수록 작은 물체를 잘 찾습니다.
# IMX500 에서는 320 또는 640 을 쓰세요. 드론용은 320 을 추천합니다(빠름).
IMGSZ = 320

# 한 번에 몇 장씩 볼지. GPU 메모리가 부족하다는 에러가 나면 줄이세요.
BATCH = 32

# 베이스 모델. yolo11n 이 가장 작고 빠릅니다 (IMX500 은 n 크기만 지원)
BASE_MODEL = 'yolo11n.pt'

# 결과물 이름
RUN_NAME = 'my_tracker'

## 4. 학습

GPU 기준 60 에폭(Epoch) 학습 시 약 10~30분 정도 소요됩니다. ☕

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=RUN_NAME,
    patience=20,        # 20 에폭 동안 나아지지 않으면 조기 종료
    plots=True,
)

# 같은 이름으로 다시 학습하면 my_tracker2, my_tracker3 ... 으로 폴더가 새로 생깁니다.
# 경로를 직접 적으면 예전 학습 결과를 보게 되므로, 방금 만들어진 폴더를 그대로 받아옵니다.
RUN_DIR = str(model.trainer.save_dir)
BEST = RUN_DIR + '/weights/best.pt'
print('\n학습 완료:', BEST)

## 5. 학습 결과 확인

평가지표에서 **mAP50이 0.7 이상**이면 실제 비행에 적용할 수 있습니다.
만약 지표가 낮다면 → 학습 이미지를 추가로 수집하거나, 라벨링에 오류가 없는지 확인하세요.

In [ ]:
from IPython.display import Image, display

metrics = YOLO(BEST).val(data=DATA_YAML, imgsz=IMGSZ)
VAL_DIR = str(metrics.save_dir)

print(f'\n검증에 사용한 가중치 : {BEST}')
print(f'mAP50    = {metrics.box.map50:.3f}   <- 0.7 이상이면 good')
print(f'mAP50-95 = {metrics.box.map:.3f}')

display(Image(RUN_DIR + '/results.png', width=900))                      # 학습 곡선
display(Image(VAL_DIR + '/confusion_matrix_normalized.png', width=600))  # 혼동행렬

In [ ]:
# 검증 이미지에 실제로 그려보기
import glob

for p in sorted(glob.glob(VAL_DIR + '/val_batch*_pred.jpg'))[:2]:
    display(Image(p, width=900))

## 6. IMX500용 모델 변환 (양자화)

Raspberry Pi AI 카메라의 IMX500 칩은 부동소수점 연산을 지원하지 않으므로, 모델을 **8비트 정수형으로 압축**해야 합니다.
이 과정을 **양자화(Quantization)**라고 하며, `data=` 파라미터로 지정한 이미지 데이터셋을 기준으로 자동 보정을 수행합니다.

**약 5~10분** 정도 소요됩니다. 중간에 실행을 중단하지 마세요.

> 시작할 때 `Exporting on CPU while CUDA is available...` 경고가 한 줄 뜨는데, GPU 로 알아서 바꿔 준다는 안내이므로 정상입니다.

In [ ]:
model = YOLO(BEST)
export_dir = model.export(format='imx', data=DATA_YAML, imgsz=IMGSZ)
print('\n변환 결과 폴더:', export_dir)

!ls -la {export_dir}

## 7. 다운로드

라즈베리파이에 적용하려면 `packerOut.zip`과 `labels.txt` 파일이 필요합니다. 아래 셀을 실행하여 하나의 압축 파일로 다운로드합니다.

In [ ]:
import os, shutil
from google.colab import files

OUT = '/content/imx500_out'
os.makedirs(OUT, exist_ok=True)

shutil.copy(os.path.join(export_dir, 'packerOut.zip'), OUT)
shutil.copy(os.path.join(export_dir, 'labels.txt'), OUT)
shutil.copy(BEST, os.path.join(OUT, 'best.pt'))   # 나중에 재학습용 백업

print('클래스 목록 (config.yaml 의 target_class 에 이 이름 중 하나를 쓰세요):')
print(open(os.path.join(OUT, 'labels.txt')).read())

shutil.make_archive(f'/content/{RUN_NAME}_imx500', 'zip', OUT)
files.download(f'/content/{RUN_NAME}_imx500.zip')

## 8. 라즈베리파이 탑재 및 모델 적용

먼저 다운로드한 ZIP 파일을 라즈베리파이로 복사합니다. **내 PC 터미널**에서 실행하세요:

```bash
scp my_tracker_imx500.zip pi@drone.local:~/
```

> `drone.local` 접속이 안 되면 라즈베리파이에서 `hostname -I` 로 IP 를 확인한 뒤
> `scp my_tracker_imx500.zip pi@192.168.x.x:~/` 형태로 실행하세요. (윈도우는 PowerShell)

이어서 **라즈베리파이 SSH 터미널**에서 다음 명령을 실행합니다:

```bash
# 1) 압축 풀기
mkdir -p ~/models && cd ~/models
unzip ~/my_tracker_imx500.zip

# 2) .rpk 로 포장  (imx500-tools 는 install.sh 에서 이미 설치됨)
imx500-package -i packerOut.zip -o .
#   -> ~/models/network.rpk 생성

# 3) 라벨 파일 옮기기
cp labels.txt ~/ai-tracking-drone/assets/my_labels.txt
```

이어서 `config.yaml` 파일의 설정을 다음과 같이 수정합니다:

```yaml
camera:
  model: /home/pi/models/network.rpk
  labels: assets/my_labels.txt

detection:
  target_class: <labels.txt 안의 이름>
  bbox_normalization: true   # YOLO 계열은 반드시 true
  bbox_order: xy             # YOLO 계열은 반드시 xy
  postprocess: ""
```

마지막으로 카메라 동작을 확인합니다:

```bash
python3 tools/check_camera.py
```

> **참고:** 새로운 모델 파일(.rpk)을 처음 로드할 때는 카메라 펌웨어 업로드 과정으로 인해 1~2분 정도 소요될 수 있습니다. 이는 정상적인 동작입니다.

---
## 트러블슈팅 (문제 해결)

| 증상 | 해결 방법 |
|---|---|
| 학습 mAP 지표가 너무 낮음 | 학습 이미지 수 부족. 클래스당 200장 이상, 다양한 각도/거리/조명 조건으로 수집 |
| 학습은 정상이나 AI 카메라에서 감지하지 못함 | 양자화에 따른 성능 손실 발생. `imgsz=640` 설정으로 다시 시도 |
| 내보내기(export) 과정에서 에러 발생 | Colab 런타임 재시작 후 6번 셀부터 다시 실행 |
| CUDA out of memory 에러 발생 | `BATCH` 파라미터 수치를 16 또는 8로 하향 조정 |
| 지상과 달리 실제 비행 중 감지 실패 | 학습 데이터에 **드론 시점(상공 탑뷰)** 사진 누락. 해당 각도의 사진을 추가하여 재학습 |